# 🟠 11 — Media Prep: The First Mile (No GPU Required)

> **Goal:** before you spend GPU time on a reasoner or generator, *validate and stage
> your media* with the four Cosmos tools that need *no GPU and no model weights*.

Notebook [05](05_tool_usage.ipynb) introduced these tools one at a time. Here we **snap
them together into one pipeline**: check the host, probe a clip, sample frames, read an
image — and **fail loudly** if an input is missing, instead of quietly moving on.

```mermaid
flowchart LR
    S["🟠 cosmos_sysinfo<br/>what am I running on?"]:::tool --> P["🟠 video_probe<br/>is the clip valid?"]:::tool
    P --> F["🟠 video_extract_frames<br/>sample frames"]:::tool
    F --> I["🟠 image_read<br/>load an image"]:::tool
    I --> R([✅ media staged<br/>ready for a Cosmos model]):::out
    classDef tool fill:#FFF3E0,stroke:#F57C00,stroke-width:2px,color:#E65100
    classDef out fill:#E0F2F1,stroke:#00897B,stroke-width:2px,color:#004D40
```

Everything here runs on a **plain laptop** — the only outside requirement is
`ffmpeg`/`ffprobe` on your `PATH` (used by `video_probe` and `video_extract_frames`).

📄 Runnable script: [`../examples/11_media_prep.py`](../examples/11_media_prep.py)

In [ ]:
# 🔧 Preflight — this cell never crashes. It just tells us what's available.
import os, sys

# 🔒 Cosmos tools confine file access to a workspace allow-list for safety, read at
# call time. Our sample media sits one level up and we write frames to /tmp, so widen
# the workspace to both. (This is how you grant the tools access to a folder — never
# wider than you need.)
os.environ["COSMOS_WORKSPACE"] = os.pathsep.join([os.path.abspath(".."), "/tmp"])

PROJECT_ROOT = os.path.abspath("..")
SAMPLE_VIDEO = os.environ.get("SAMPLE_VIDEO", os.path.join(PROJECT_ROOT, "sample.mp4"))
SAMPLE_IMAGE = os.environ.get("SAMPLE_IMAGE", os.path.join(PROJECT_ROOT, "sample.png"))
FRAMES_DIR   = os.path.join("/tmp", "cosmos_media_prep_frames")

import shutil
def have(cmd): return shutil.which(cmd) is not None

print("🛠  ffmpeg  :", have("ffmpeg"))
print("🛠  ffprobe :", have("ffprobe"), "(video_probe / video_extract_frames need this)")
print("📹 sample.mp4 :", os.path.exists(SAMPLE_VIDEO))
print("🖼  sample.png :", os.path.exists(SAMPLE_IMAGE))
print("\nNo GPU needed — every cell below runs on a plain laptop.")

## Step 1 — `cosmos_sysinfo`: what am I running on?

The right first call on any new machine. It answers *"is a heavy model even runnable
here?"* before you try to load one. Works everywhere, no GPU required.

In [ ]:
from strands_cosmos import cosmos_sysinfo

r = cosmos_sysinfo()
print("status:", r["status"])
# Print the first lines — host + GPU summary
print("\n".join(r["content"][0]["text"].splitlines()[:12]))

## Step 2 — `video_probe`: is the clip valid, and what's in it?

Resolution, fps, duration, codec — the facts you need *before* deciding an fps to
sample at. If the file is missing or unreadable, the tool returns `status="error"`
instead of raising, so a pipeline can decide what to do.

In [ ]:
from strands_cosmos import video_probe

if os.path.exists(SAMPLE_VIDEO):
    r = video_probe(video_path=SAMPLE_VIDEO)
    print("status:", r["status"])
    print(r["content"][0]["text"])
else:
    print("⏭  sample.mp4 not found — set SAMPLE_VIDEO to a real clip to run this.")

## Step 3 — `video_extract_frames`: sample frames to disk

Pull evenly-spaced frames out of the clip. `fps=1.0` gives one frame per second;
`max_frames` caps the count so a long video stays cheap. These JPEGs are exactly what
you would hand to a vision model as `<image>` inputs.

In [ ]:
from strands_cosmos import video_extract_frames

if os.path.exists(SAMPLE_VIDEO):
    r = video_extract_frames(video_path=SAMPLE_VIDEO, output_dir=FRAMES_DIR,
                             fps=1.0, max_frames=8)
    print("status:", r["status"])
    print(r["content"][0]["text"])
    frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith(".jpg"))
    print("frames on disk:", len(frames), "->", frames[:3], "...")
else:
    print("⏭  sample.mp4 not found.")

## Step 4 — `image_read`: load an image so an agent can *see* it

Reads an image (confined to the workspace allow-list) and embeds it in the tool result
as a base64 image block — the shape an agent consumes. Note the three content blocks:
`text` (a summary), `json` (metadata), and `image` (the pixels).

In [ ]:
from strands_cosmos import image_read

if os.path.exists(SAMPLE_IMAGE):
    r = image_read(image_path=SAMPLE_IMAGE)
    print("status:", r["status"])
    print(r["content"][0]["text"])
    print("content blocks:", [next(iter(b)) for b in r["content"]])
else:
    print("⏭  sample.png not found — set SAMPLE_IMAGE to a real image to run this.")

## Putting it together — a fail-loud pipeline

The script version ([`11_media_prep.py`](../examples/11_media_prep.py)) chains these four
steps and **stops with a non-zero exit if any input is missing** — so it drops straight
into CI or an edge-triage script as a gate before GPU work:

```python
def _check(result, label):
    print(result["content"][0]["text"].rstrip())
    if result.get("status") != "success":
        raise SystemExit(1)   # fail loud — don't pretend the media was staged
    return result
```

Run it end to end:

```bash
python examples/11_media_prep.py
# or point it at your own media:
SAMPLE_VIDEO=/path/clip.mp4 SAMPLE_IMAGE=/path/frame.png python examples/11_media_prep.py
```

# 🎓 What you learned

| Tool | Needs GPU? | Answers |
|------|:---------:|---------|
| `cosmos_sysinfo` | ❌ | What hardware am I on? Can a model run here? |
| `video_probe` | ❌ | Is the clip valid — resolution / fps / duration / codec? |
| `video_extract_frames` | ❌ | Sample frames → JPEGs for `<image>` inputs |
| `image_read` | ❌ | Load an image into the block shape an agent consumes |

These four are the **first mile** of every vision pipeline. Validate here — cheaply, on a
laptop — before spending GPU time on a reasoner or generator.

**Next:** [01 — Basic Text](01_basic_text.ipynb) to start reasoning, or
[04 — Embodied Reasoning](04_embodied_reasoning.ipynb) to feed one of these frames to a
model. 🚀